# Requisitos

In [1]:
!pip install torch torchvision torchaudio
!pip install numpy pandas scikit-learn optuna
!pip install scikit-fuzzy

   ---------------------------------------- 0.0/2.1 MB ? eta -:--:--
   -------------- ------------------------- 0.8/2.1 MB 5.6 MB/s eta 0:00:01
   -------------- ------------------------- 0.8/2.1 MB 5.6 MB/s eta 0:00:01
   ------------------------ --------------- 1.3/2.1 MB 2.5 MB/s eta 0:00:01
   ------------------------ --------------- 1.3/2.1 MB 2.5 MB/s eta 0:00:01
   ----------------------------- ---------- 1.6/2.1 MB 1.5 MB/s eta 0:00:01
   ----------------------------- ---------- 1.6/2.1 MB 1.5 MB/s eta 0:00:01
   ----------------------------- ---------- 1.6/2.1 MB 1.5 MB/s eta 0:00:01
   ---------------------------------------- 2.1/2.1 MB 1.2 MB/s  0:00:01

   ---------------------------------------- 0/6 [Mako]
   ---------------------------------------- 0/6 [Mako]
   ---------------------------------------- 0/6 [Mako]
   ------ --------------------------------- 1/6 [greenlet]
   ------ --------------------------------- 1/6 [greenlet]
   ------ --------------------------------

In [7]:
import sys
print(sys.executable)
print(sys.version)


C:\Users\caio.grasso\AppData\Local\miniconda3\envs\dl311\python.exe
3.11.14 | packaged by Anaconda, Inc. | (main, Oct 21 2025, 18:30:03) [MSC v.1929 64 bit (AMD64)]


In [11]:
!python -m pip show optuna

Name: optuna
Version: 4.6.0
Summary: A hyperparameter optimization framework
Home-page: https://optuna.org/
Author: Takuya Akiba
Author-email: 
License: 
Location: C:\Users\caio.grasso\AppData\Local\miniconda3\Lib\site-packages
Requires: alembic, colorlog, numpy, packaging, PyYAML, sqlalchemy, tqdm
Required-by: 


In [1]:
import os
import time
import json
import random
from pathlib import Path
from typing import Dict, Any, Tuple

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split

from torchvision import datasets, transforms, models

from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

import optuna

# 1.1. Configurações de device e seeds
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Usando device:", DEVICE)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# 1.2. Pastas
BASE_DIR = Path(".")
DATA_DIR = BASE_DIR / "data"
RESULTS_DIR = BASE_DIR / "results"
RESULTS_DIR.mkdir(exist_ok=True, parents=True)

Usando device: cuda


C:\Users\caio.grasso\AppData\Local\miniconda3\envs\dl311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# 2.1. Transforms padrão para classificação de imagens (32x32, CIFAR)
IMG_SIZE = 224  # redimensionar para encaixar nas CNNs padrão (ResNet, VGG, MobileNet)

train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomCrop(IMG_SIZE, padding=4),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

test_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])


def get_dataloaders(
    dataset_name: str,
    batch_size: int,
    val_split: float = 0.1,
    num_workers: int = 2,
) -> Tuple[DataLoader, DataLoader, DataLoader, int]:
    """
    Retorna dataloaders (train, val, test) e número de classes
    para o dataset especificado.
    """
    if dataset_name.lower() == "cifar10":
        train_dataset = datasets.CIFAR10(
            root=DATA_DIR, train=True, download=True, transform=train_transform
        )
        test_dataset = datasets.CIFAR10(
            root=DATA_DIR, train=False, download=True, transform=test_transform
        )
        num_classes = 10

    elif dataset_name.lower() == "cifar100":
        train_dataset = datasets.CIFAR100(
            root=DATA_DIR, train=True, download=True, transform=train_transform
        )
        test_dataset = datasets.CIFAR100(
            root=DATA_DIR, train=False, download=True, transform=test_transform
        )
        num_classes = 100

    else:
        raise ValueError(f"Dataset {dataset_name} não implementado ainda.")

    # Split train/val
    val_size = int(len(train_dataset) * val_split)
    train_size = len(train_dataset) - val_size
    train_subset, val_subset = random_split(
        train_dataset,
        [train_size, val_size],
        generator=torch.Generator().manual_seed(SEED),
    )

    train_loader = DataLoader(
        train_subset,
        batch_size=batch_size,
        shuffle=True,
        num_workers=num_workers,
        pin_memory=torch.cuda.is_available(),
    )
    val_loader = DataLoader(
        val_subset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
        pin_memory=torch.cuda.is_available(),
    )
    test_loader = DataLoader(
        test_dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
        pin_memory=torch.cuda.is_available(),
    )

    return train_loader, val_loader, test_loader, num_classes


In [3]:
from torchvision.datasets import ImageFolder

def get_kaggle_imagefolder_dataloaders(root_dir: str, batch_size: int, val_split: float = 0.1):
    root_path = Path(root_dir)
    train_path = root_path / "train"
    test_path = root_path / "test"   # ou "val", depende do dataset

    full_train = ImageFolder(train_path, transform=train_transform)
    test_ds = ImageFolder(test_path, transform=test_transform)

    val_size = int(len(full_train) * val_split)
    train_size = len(full_train) - val_size
    train_ds, val_ds = random_split(
        full_train,
        [train_size, val_size],
        generator=torch.Generator().manual_seed(SEED)
    )

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False)
    test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False)
    num_classes = len(full_train.dataset.classes) if hasattr(full_train, "dataset") else len(full_train.classes)

    return train_loader, val_loader, test_loader, num_classes

In [4]:
def create_model(model_name: str, num_classes: int) -> nn.Module:
    model_name = model_name.lower()
    
    if model_name == "resnet18":
        model = models.resnet18(pretrained=True)
        in_features = model.fc.in_features
        model.fc = nn.Linear(in_features, num_classes)

    elif model_name == "resnet50":
        model = models.resnet50(pretrained=True)
        in_features = model.fc.in_features
        model.fc = nn.Linear(in_features, num_classes)

    elif model_name == "vgg16":
        model = models.vgg16_bn(pretrained=True)
        in_features = model.classifier[-1].in_features
        model.classifier[-1] = nn.Linear(in_features, num_classes)

    elif model_name == "mobilenet_v2":
        model = models.mobilenet_v2(pretrained=True)
        in_features = model.classifier[-1].in_features
        model.classifier[-1] = nn.Linear(in_features, num_classes)

    else:
        raise ValueError(f"Modelo {model_name} não suportado.")

    return model.to(DEVICE)

In [5]:
class FuzzyAggregator:
    """
    Agregador fuzzy multicritério (Sugeno) para avaliar um conjunto de hiperparâmetros.

    Entradas esperadas:
      - acc: acurácia (0 a 1)
      - f1: F1-score (0 a 1)
      - train_time_norm: tempo de treinamento normalizado em [0, 1] (0 = muito rápido, 1 = muito lento)
      - infer_time_norm: tempo de inferência normalizado em [0, 1]
    Saída:
      - quality_score: escalar em [0, 1]
    """

    def __init__(
        self,
        max_train_time_ref: float = 60.0,
        max_infer_time_ref: float = 0.02,
    ):
        self.max_train_time_ref = max_train_time_ref
        self.max_infer_time_ref = max_infer_time_ref

    # ---- Funções de pertinência básicas ----
    @staticmethod
    def _triangular(x, a, b, c):
        return max(min((x - a) / (b - a + 1e-8), (c - x) / (c - b + 1e-8)), 0.0)

    def mu_acc_low(self, x):
        return self._triangular(x, 0.0, 0.2, 0.5)

    def mu_acc_med(self, x):
        return self._triangular(x, 0.3, 0.5, 0.7)

    def mu_acc_high(self, x):
        return self._triangular(x, 0.6, 0.8, 1.0)

    # Usar as mesmas para F1
    mu_f1_low = mu_acc_low
    mu_f1_med = mu_acc_med
    mu_f1_high = mu_acc_high

    # Para tempo normalizado (0 = rápido, 1 = lento)
    def mu_time_fast(self, x):
        return self._triangular(x, 0.0, 0.0, 0.4)

    def mu_time_medium(self, x):
        return self._triangular(x, 0.3, 0.5, 0.7)

    def mu_time_slow(self, x):
        return self._triangular(x, 0.6, 1.0, 1.0)

    # ---- Normalização de tempos ----
    def normalize_train_time(self, train_time: float) -> float:
        return float(np.clip(train_time / self.max_train_time_ref, 0.0, 1.0))

    def normalize_infer_time(self, infer_time: float) -> float:
        return float(np.clip(infer_time / self.max_infer_time_ref, 0.0, 1.0))

    # ---- Avaliação das regras e defuzzificação (Sugeno) ----
    def compute_quality(
        self,
        acc: float,
        f1: float,
        train_time: float,
        infer_time: float,
    ) -> float:
        # Normalizar tempos
        t_train = self.normalize_train_time(train_time)
        t_infer = self.normalize_infer_time(infer_time)

        # Memberships
        acc_L = self.mu_acc_low(acc)
        acc_M = self.mu_acc_med(acc)
        acc_H = self.mu_acc_high(acc)

        f1_L = self.mu_f1_low(self, f1)
        f1_M = self.mu_f1_med(self, f1)
        f1_H = self.mu_f1_high(self, f1)

        tr_F = self.mu_time_fast(t_train)
        tr_M = self.mu_time_medium(t_train)
        tr_S = self.mu_time_slow(t_train)

        inf_F = self.mu_time_fast(t_infer)
        inf_M = self.mu_time_medium(t_infer)
        inf_S = self.mu_time_slow(t_infer)

        # Regras (Sugeno: cada regra produz um valor crisp z_i)
        rules = []

        # R1: SE acc é Alta E f1 é Alta E treino é Rápido E inferência é Rápida -> Qualidade = 0.98
        w1 = min(acc_H, f1_H, tr_F, inf_F)
        z1 = 0.98
        rules.append((w1, z1))

        # R2: SE acc é Alta E f1 é Alta E inferência é Média -> Qualidade = 0.9
        w2 = min(acc_H, f1_H, inf_M)
        z2 = 0.90
        rules.append((w2, z2))

        # R3: SE acc é Alta E f1 é Média -> Qualidade = 0.8
        w3 = min(acc_H, f1_M)
        z3 = 0.80
        rules.append((w3, z3))

        # R4: SE acc é Média E f1 é Média -> Qualidade = 0.6
        w4 = min(acc_M, f1_M)
        z4 = 0.60
        rules.append((w4, z4))

        # R5: SE acc é Baixa OU f1 é Baixa -> Qualidade = 0.3
        w5 = max(acc_L, f1_L)
        z5 = 0.30
        rules.append((w5, z5))

        # R6: SE treino é Muito Lento OU inferência é Muito Lenta -> Qualidade = 0.4 (penalização)
        w6 = max(tr_S, inf_S)
        z6 = 0.40
        rules.append((w6, z6))

        # R7: fallback moderado
        w7 = 0.1  # pequena ativação base
        z7 = 0.5
        rules.append((w7, z7))

        # Defuzzificação Sugeno: média ponderada
        numerator = sum(w * z for w, z in rules)
        denominator = sum(w for w, _ in rules) + 1e-8

        quality = numerator / denominator
        return float(np.clip(quality, 0.0, 1.0))


# Instância global do agregador (ajuste os referenciais de tempo conforme seu ambiente)
fuzzy_aggregator = FuzzyAggregator(
    max_train_time_ref=120.0,   # s por época, por exemplo
    max_infer_time_ref=0.05,    # s por batch de inferência, por exemplo
)


In [6]:
def train_one_epoch(
    model: nn.Module,
    loader: DataLoader,
    criterion: nn.Module,
    optimizer: optim.Optimizer,
) -> Tuple[float, float]:
    model.train()
    running_loss = 0.0
    start_time = time.perf_counter()

    for images, labels in loader:
        images = images.to(DEVICE)
        labels = labels.to(DEVICE)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)

    end_time = time.perf_counter()
    epoch_time = end_time - start_time
    avg_loss = running_loss / len(loader.dataset)

    return avg_loss, epoch_time


def evaluate_model(model: nn.Module, loader: DataLoader) -> Dict[str, float]:
    model.eval()
    all_preds = []
    all_labels = []

    start_time = time.perf_counter()
    with torch.no_grad():
        for images, labels in loader:
            images = images.to(DEVICE)
            labels = labels.to(DEVICE)

            outputs = model(images)
            preds = torch.argmax(outputs, dim=1)

            all_preds.extend(preds.cpu().numpy().tolist())
            all_labels.extend(labels.cpu().numpy().tolist())

    end_time = time.perf_counter()
    infer_time = end_time - start_time
    infer_time_per_sample = infer_time / len(loader.dataset)

    acc = accuracy_score(all_labels, all_preds)
    f1 = f1_score(all_labels, all_preds, average="macro")
    prec = precision_score(all_labels, all_preds, average="macro", zero_division=0)
    rec = recall_score(all_labels, all_preds, average="macro")

    return {
        "accuracy": acc,
        "f1": f1,
        "precision": prec,
        "recall": rec,
        "infer_time": infer_time,
        "infer_time_per_sample": infer_time_per_sample,
    }


In [7]:
def objective(
    trial: optuna.Trial,
    dataset_name: str,
    model_name: str,
    max_epochs: int = 5,
) -> float:
    # Hiperparâmetros
    batch_size = trial.suggest_categorical("batch_size", [32, 64, 128])
    lr = trial.suggest_loguniform("lr", 1e-4, 1e-2)
    weight_decay = trial.suggest_loguniform("weight_decay", 1e-6, 1e-3)
    optimizer_name = trial.suggest_categorical("optimizer", ["sgd", "adam"])
    # Opcional: dropout, data augmentation extra etc.

    # Dataloaders
    train_loader, val_loader, test_loader, num_classes = get_dataloaders(
        dataset_name, batch_size=batch_size
    )

    # Modelo
    model = create_model(model_name, num_classes=num_classes)

    criterion = nn.CrossEntropyLoss()
    if optimizer_name == "sgd":
        optimizer = optim.SGD(
            model.parameters(), lr=lr, momentum=0.9, weight_decay=weight_decay
        )
    else:
        optimizer = optim.Adam(
            model.parameters(), lr=lr, weight_decay=weight_decay
        )

    # Loop de treino reduzido (para busca de hiperparâmetros)
    total_train_time = 0.0
    for epoch in range(max_epochs):
        train_loss, epoch_time = train_one_epoch(
            model, train_loader, criterion, optimizer
        )
        total_train_time += epoch_time

        # (Opcional) early stopping bem simples
        if epoch == 0 and train_loss > 3.0:
            # se estiver muito ruim, aborta cedo
            break

    avg_train_time_per_epoch = total_train_time / max(max_epochs, 1)

    # Avaliar no conjunto de validação
    val_metrics = evaluate_model(model, val_loader)

    acc = val_metrics["accuracy"]
    f1 = val_metrics["f1"]
    infer_time = val_metrics["infer_time"]  # tempo total de inferência val

    # Calcular score fuzzy
    fuzzy_score = fuzzy_aggregator.compute_quality(
        acc=acc,
        f1=f1,
        train_time=avg_train_time_per_epoch,
        infer_time=infer_time / len(val_loader),  # tempo médio por batch
    )

    # Log adicional no trial
    trial.set_user_attr("val_metrics", val_metrics)
    trial.set_user_attr("avg_train_time_per_epoch", avg_train_time_per_epoch)

    return fuzzy_score


In [8]:
SEARCH_CONFIG = {
    "cifar10": ["resnet18", "mobilenet_v2", "vgg16"],
    "cifar100": ["resnet18", "mobilenet_v2"],
    # Para Kaggle depois: "meu_kaggle": ["resnet50", ...]
}

N_TRIALS = 10  # aumente para algo como 30–50 nas execuções finais
MAX_EPOCHS_SEARCH = 5  # aumente para 15–30 depois

summary_results = []

for dataset_name, model_list in SEARCH_CONFIG.items():
    for model_name in model_list:
        print(f"\n=== Otimizando {model_name} em {dataset_name} ===")

        study_name = f"{dataset_name}_{model_name}_fuzzy_opt"
        storage_url = f"sqlite:///{RESULTS_DIR / (study_name + '.db')}"  # opcional

        study = optuna.create_study(
            study_name=study_name,
            direction="maximize",
            storage=storage_url,
            load_if_exists=True,
        )

        study.optimize(
            lambda trial: objective(
                trial, dataset_name=dataset_name, model_name=model_name, max_epochs=MAX_EPOCHS_SEARCH
            ),
            n_trials=N_TRIALS,
        )

        best_trial = study.best_trial

        print(f"Melhor score fuzzy: {best_trial.value:.4f}")
        print("Melhores hiperparâmetros:", best_trial.params)

        # Guardar em memória
        summary_results.append({
            "dataset": dataset_name,
            "model": model_name,
            "best_score": best_trial.value,
            "best_params": best_trial.params,
            "best_val_metrics": best_trial.user_attrs.get("val_metrics", {}),
            "avg_train_time_per_epoch": best_trial.user_attrs.get("avg_train_time_per_epoch", None),
        })

        # Salvar em JSON para reuso posterior
        out_file = RESULTS_DIR / f"best_params_{dataset_name}_{model_name}.json"
        with out_file.open("w") as f:
            json.dump(summary_results[-1], f, indent=4)



=== Otimizando resnet18 em cifar10 ===


[I 2025-12-11 12:50:23,615] A new study created in RDB with name: cifar10_resnet18_fuzzy_opt
C:\Users\caio.grasso\AppData\Local\Temp\ipykernel_4572\1979219645.py:9: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  lr = trial.suggest_loguniform("lr", 1e-4, 1e-2)
C:\Users\caio.grasso\AppData\Local\Temp\ipykernel_4572\1979219645.py:10: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  weight_decay = trial.suggest_loguniform("weight_decay", 1e-6, 1e-3)


100%|███████████████████████████████████████████████████████████████████████████████| 170M/170M [00:30<00:00, 5.52MB/s]


Extracting data\cifar-10-python.tar.gz to data
Files already downloaded and verified


C:\Users\caio.grasso\AppData\Local\miniconda3\envs\dl311\Lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
C:\Users\caio.grasso\AppData\Local\miniconda3\envs\dl311\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
[W 2025-12-11 13:02:17,508] Trial 0 failed with parameters: {'batch_size': 32, 'lr': 0.001614664819815868, 'weight_decay': 0.0004933789487867447, 'optimizer': 'adam'} because of the following error: KeyboardInterrupt().
Traceback (most recent call last):
  File "C:\Users\caio.grasso\AppData\Local\miniconda

KeyboardInterrupt: 

In [ ]:
df_summary = pd.DataFrame([
    {
        "dataset": r["dataset"],
        "model": r["model"],
        "best_score": r["best_score"],
        "acc_val": r["best_val_metrics"].get("accuracy", None),
        "f1_val": r["best_val_metrics"].get("f1", None),
        "prec_val": r["best_val_metrics"].get("precision", None),
        "rec_val": r["best_val_metrics"].get("recall", None),
        "avg_train_time_per_epoch": r["avg_train_time_per_epoch"],
        "hyperparams": r["best_params"],
    }
    for r in summary_results
])

df_summary
